# AI Research Paper Q&A - Generation Evaluation (Ragas)

## Layer 1: compares BASE Qwen3-4B-Instruct vs your fine-tuned adapter on real QASPER test questions.

Metrics: faithfulness, context precision, answer relevancy, hallucination rate.

Run on a free Colab T4 GPU. Before running: upload `test.jsonl` (from your local
`data/` folder) into this Colab session's file browser.

In [ ]:
!pip install -q -U "transformers>=4.51.0,<5.0.0" "trl==0.19.1" peft accelerate bitsandbytes
!pip install -q -U datasets "huggingface_hub<1.0,>=0.34.0" "pandas<3" "ragas==0.3.9" langchain-google-genai langchain-groq
# Re-pin: downstream packages above can silently bump these past what
# transformers and Colab's own preinstalled packages support.
!pip install -q -U "huggingface_hub<1.0,>=0.34.0" "pandas<3"

In [ ]:
import os
import gc
import json

import torch
import pandas as pd
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, ContextPrecision, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.rate_limiters import InMemoryRateLimiter

In [ ]:
# Constants

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
HF_USER = "your-hf-username"  # <-- change this to your HuggingFace username
FINETUNED_ADAPTER = f"{HF_USER}/qwen3-4b-ai-research-qa-v2"

# Keep this small at first - Groq's free tier allows 25 requests/minute for
# gpt-oss-120b, and each example triggers multiple judge calls (one per
# metric), so 50 examples already takes a while at that rate. Raise it later
# for a more statistically solid comparison once you've confirmed it works.
NUM_EVAL_SAMPLES = 50

MAX_NEW_TOKENS = 300
SEED = 42

SYSTEM_PROMPT = (
    "You are an expert AI research assistant. Answer the question based "
    "only on the provided context. Be precise, technical, and cite "
    "specific details from the context."
)

### Log in to HuggingFace and set your judge API keys

Click the key icon on the left sidebar -> add secrets named `HF_TOKEN` (your HF token),
`GOOGLE_API_KEY` (free key from https://aistudio.google.com/apikey - used only for
embeddings), and `GROQ_API_KEY` (free key from https://console.groq.com/keys - used
for the judge LLM).

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

### Load the test set

Upload `test.jsonl` into this Colab session first (left sidebar -> folder icon -> upload).

In [ ]:
test_dataset = load_dataset("json", data_files={"test": "test.jsonl"})["test"]
test_dataset = test_dataset.shuffle(seed=SEED).select(range(min(NUM_EVAL_SAMPLES, len(test_dataset))))
print(f"Evaluating on {len(test_dataset)} test examples")

### Generation helper

Same chat template used in training, so we're evaluating the model the exact
way it will actually be called in the RAG pipeline.

In [ ]:
def generate_answers(model, tokenizer, examples):
    answers = []
    model.eval()
    for ex in examples:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {ex['context']}\nQuestion: {ex['question']}"},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated = output_ids[0][input_ids.shape[-1]:]
        answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
        answers.append(answer)
    return answers

### Quantization config (same 4-bit NF4 setup used in training)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### Generate answers with the BASE model

In [ ]:
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto"
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Generating answers with base model...")
base_answers = generate_answers(base_model, tokenizer, test_dataset)

del base_model
gc.collect()
torch.cuda.empty_cache()
print("Base model freed from memory.")

### Generate answers with the FINE-TUNED model

In [ ]:
print("Loading base model + fine-tuned adapter...")
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto"
)
ft_model = PeftModel.from_pretrained(ft_base, FINETUNED_ADAPTER)
ft_model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Generating answers with fine-tuned model...")
finetuned_answers = generate_answers(ft_model, tokenizer, test_dataset)

del ft_model, ft_base
gc.collect()
torch.cuda.empty_cache()
print("Fine-tuned model freed from memory.")

### Set up the Ragas judge (Groq gpt-oss-120b, free tier)

Ragas metrics work by having an LLM "judge" each generated answer against the
context and reference answer. This is separate from the model being evaluated.

Groq's free tier for `openai/gpt-oss-120b` allows 25 requests/minute, so we
attach a rate limiter that throttles ALL calls through this one client to stay
under that - Ragas will queue and wait rather than firing requests too fast.
Embeddings still go through Gemini (that part wasn't the bottleneck).

If `openai/gpt-oss-120b` ever errors on structured output, swap the model
string below for `llama-3.3-70b-versatile` as a fallback.

In [ ]:
# 25 requests/minute = ~0.4/sec. Set slightly under with no burst allowance
# so we never trip the free-tier limit even with Ragas' internal retries.
groq_rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.4,
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

judge_llm = LangchainLLMWrapper(
    ChatGroq(model="openai/gpt-oss-120b", temperature=0, rate_limiter=groq_rate_limiter)
)
judge_embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"))

metrics = [
    Faithfulness(),
    ContextPrecision(),
    ResponseRelevancy(),
]

# Keep Ragas' own concurrency low too - the rate limiter throttles total
# throughput, but a low worker count avoids a burst of calls all queuing at
# once and timing out while they wait.
eval_run_config = RunConfig(max_workers=3, timeout=180)

### Build Ragas evaluation datasets and run scoring

In [ ]:
def build_ragas_dataset(examples, answers):
    rows = []
    for ex, answer in zip(examples, answers):
        rows.append({
            "user_input": ex["question"],
            "retrieved_contexts": [ex["context"]],
            "response": answer,
            "reference": ex["answer"],
        })
    return EvaluationDataset.from_list(rows)

base_ragas_ds = build_ragas_dataset(test_dataset, base_answers)
finetuned_ragas_ds = build_ragas_dataset(test_dataset, finetuned_answers)

In [ ]:
print("Scoring BASE model answers with Ragas (this calls the judge LLM per example)...")
base_results = evaluate(dataset=base_ragas_ds, metrics=metrics, llm=judge_llm, embeddings=judge_embeddings, run_config=eval_run_config)
base_df = base_results.to_pandas()
base_df.to_csv("base_model_ragas_results.csv", index=False)
base_df.head()

In [ ]:
print("Scoring FINE-TUNED model answers with Ragas...")
finetuned_results = evaluate(dataset=finetuned_ragas_ds, metrics=metrics, llm=judge_llm, embeddings=judge_embeddings, run_config=eval_run_config)
finetuned_df = finetuned_results.to_pandas()
finetuned_df.to_csv("finetuned_model_ragas_results.csv", index=False)
finetuned_df.head()

### Summary comparison table

Hallucination rate is reported as `1 - faithfulness`, since faithfulness
already measures the fraction of claims in the answer that are actually
supported by the given context - its complement is a natural hallucination
rate (Ragas does not ship a separately-named hallucination metric).

In [ ]:
def summarize(df, label):
    faithfulness = df["faithfulness"].mean()
    return {
        "model": label,
        "faithfulness": round(faithfulness, 4),
        "context_precision": round(df["context_precision"].mean(), 4),
        "answer_relevancy": round(df["answer_relevancy"].mean(), 4),
        "hallucination_rate": round(1 - faithfulness, 4),
    }

summary_df = pd.DataFrame([
    summarize(base_df, "Base Qwen3-4B-Instruct"),
    summarize(finetuned_df, "Fine-tuned Qwen3-4B (v2)"),
])
summary_df.to_csv("generation_eval_summary.csv", index=False)
summary_df

### Download your results

Grab `generation_eval_summary.csv` from the file browser on the left - this is
the table that goes into your README's evaluation results section.